# EDRAS · Train the MRI dementia-stage CNN

This notebook trains a real image model (EfficientNet-B0, transfer learning) on your Kaggle MRI datasets and exports
`dementia_cnn.onnx` + `dementia_cnn_meta.json` for the EDRAS backend.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

What it does differently from the old model:
- Looks at the actual brain image (a CNN), not 17 brightness statistics.
- **Splits by patient**, so no person appears in both training and test. The accuracy you get is honest.
- Reports slice-level **and** scan-level accuracy, a confusion matrix, and per-class recall.

Time on a free T4 GPU: about 30–45 minutes.

## 1 · Settings

In [ ]:
# ---- Edit these if your datasets differ ----
OASIS_DATASET   = "ninadaithal/imagesoasis"                 # 2024 OASIS slices (has patient IDs in filenames)
EXTRA_DATASET   = "tourist55/alzheimers-dataset-4-class-of-images"  # 2021-22 4-class set (optional)
USE_EXTRA       = True     # add the extra dataset to TRAINING only (it has no patient IDs, so it never goes in test)

EPOCHS          = 6        # 6 is enough with a pretrained backbone; raise to 10 if you have time
IMG_SIZE        = 224
BATCH_SIZE      = 64
SAMPLES_PER_EPOCH = 24000  # balanced sampling per epoch (keeps epochs ~4-5 min on a T4)
SLICE_RANGE     = (100, 160)  # OASIS axial slice indices to keep (mid-brain, where hippocampus/ventricles are)
SEED            = 42

import os
TEST_MODE = os.environ.get("EDRAS_TEST") == "1"   # used only for local smoke tests
if TEST_MODE: SAMPLES_PER_EPOCH, BATCH_SIZE, IMG_SIZE = 128, 8, 128
DATA_ROOT = os.environ.get("EDRAS_DATA_ROOT", "/content/data")
OUT_DIR   = os.environ.get("EDRAS_OUT", "/content/edras_model")
os.makedirs(OUT_DIR, exist_ok=True)

## 2 · Install & check GPU

In [ ]:
import subprocess, sys
if not TEST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "onnx", "onnxruntime", "onnxscript"], check=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (enable T4 in Runtime settings!)")

## 3 · Download the datasets
`kagglehub` will ask you to log in to Kaggle the first time (paste your Kaggle username + API key from
kaggle.com → Settings → API → *Create New Token*).

In [ ]:
from pathlib import Path
if TEST_MODE:
    oasis_path = Path(DATA_ROOT) / "oasis"
    extra_path = Path(DATA_ROOT) / "extra"
else:
    import kagglehub
    kagglehub.login()
    oasis_path = Path(kagglehub.dataset_download(OASIS_DATASET))
    extra_path = Path(kagglehub.dataset_download(EXTRA_DATASET)) if USE_EXTRA else None
print("OASIS:", oasis_path)
print("Extra:", extra_path)

## 4 · Index images, labels and patient IDs

In [ ]:
import re, random
import numpy as np
import pandas as pd

CLASSES = ["Non Demented", "Very Mild Dementia", "Mild Dementia", "Moderate Dementia"]
SEVERITY = [5.0, 35.0, 65.0, 92.0]   # used to turn probabilities into a 0-100 risk index
EXTS = {".jpg", ".jpeg", ".png"}

def label_from_folder(name):
    n = re.sub(r"[^a-z]", "", name.lower())   # 'Very mild Dementia' / 'VeryMildDemented' -> 'verymild...' 
    if n.startswith("non") or "nodementia" in n: return 0
    if "verymild" in n: return 1
    if "moderate" in n: return 3
    if "mild" in n: return 2
    return None

def index_folder(root, source):
    rows = []
    for p in Path(root).rglob("*"):
        if p.suffix.lower() not in EXTS: continue
        lab = None
        for parent in p.parents:            # nearest folder whose name is a class
            lab = label_from_folder(parent.name)
            if lab is not None: break
            if parent == Path(root): break
        if lab is None: continue
        m = re.search(r"(OAS\d?_\d{4})_MR(\d)", p.name)
        patient = m.group(1) if m else None
        scan = f"{m.group(1)}_MR{m.group(2)}" if m else None
        sl = re.search(r"_(\d{2,3})\.(jpe?g|png)$", p.name, re.I)
        rows.append({"path": str(p), "label": lab, "patient": patient, "scan": scan,
                     "slice": int(sl.group(1)) if sl else None, "source": source})
    return pd.DataFrame(rows)

oasis = index_folder(oasis_path, "oasis")
assert len(oasis), "No OASIS images found - check OASIS_DATASET"
oasis = oasis[oasis.patient.notna()]
in_range = oasis.slice.between(*SLICE_RANGE) | oasis.slice.isna()
oasis = oasis[in_range].reset_index(drop=True)

extra = index_folder(extra_path, "extra") if (USE_EXTRA and extra_path) else pd.DataFrame(columns=oasis.columns)

print("OASIS images:", len(oasis), "| patients:", oasis.patient.nunique(), "| scans:", oasis.scan.nunique())
print(oasis.groupby("label").agg(images=("path", "size"), patients=("patient", "nunique")).rename(index=dict(enumerate(CLASSES))))
print("Extra images:", len(extra))
if len(extra): print(extra.label.value_counts().rename(index=dict(enumerate(CLASSES))))

## 5 · Patient-wise split (no leakage)
Every patient goes entirely into train, validation **or** test. The test set is OASIS only, because the extra dataset
has no patient IDs and may contain the same people.

In [ ]:
rng = random.Random(SEED)
# A patient's label = their most severe scan label (a patient can have several visits)
patient_label = oasis.groupby("patient").label.max()
split = {}
for lab in range(4):
    pts = sorted(patient_label[patient_label == lab].index)
    rng.shuffle(pts)
    n = len(pts)
    n_test = max(1, round(n * 0.15)) if n >= 3 else (1 if n == 2 else 0)
    n_val  = max(1, round(n * 0.15)) if n >= 4 else 0
    for i, p in enumerate(pts):
        split[p] = "test" if i < n_test else "val" if i < n_test + n_val else "train"

oasis["split"] = oasis.patient.map(split)
extra["split"] = "train"
df = pd.concat([oasis, extra], ignore_index=True)

for s in ["train", "val", "test"]:
    d = df[df.split == s]
    print(f"{s:5s}: {len(d):6d} images, {d.patient.nunique():4d} patients |",
          {CLASSES[k]: int(v) for k, v in d.label.value_counts().sort_index().items()})
assert not (set(df[df.split=='train'].patient.dropna()) & set(df[df.split=='test'].patient.dropna())), "leak!"
print("No patient appears in both train and test ✓")

## 6 · Data pipeline

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.92, 1.08)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Grayscale(num_output_channels=3), T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class MRIDataset(Dataset):
    def __init__(self, frame, tf): self.f, self.tf = frame.reset_index(drop=True), tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        return self.tf(Image.open(r.path).convert("L")), int(r.label)

train_df, val_df, test_df = (df[df.split == s] for s in ["train", "val", "test"])

# Balanced sampling: rare classes (Moderate) are drawn as often as common ones
counts = train_df.label.value_counts().reindex(range(4), fill_value=0).values
w = 1.0 / np.maximum(counts, 1)
sampler = WeightedRandomSampler(torch.tensor(w[train_df.label.values], dtype=torch.double),
                                num_samples=min(SAMPLES_PER_EPOCH, len(train_df) * 4), replacement=True)
workers = 0 if TEST_MODE else 2
train_dl = DataLoader(MRIDataset(train_df, train_tf), batch_size=BATCH_SIZE, sampler=sampler, num_workers=workers, pin_memory=DEVICE=="cuda")
val_dl   = DataLoader(MRIDataset(val_df, eval_tf),  batch_size=BATCH_SIZE * 2, num_workers=workers)
test_dl  = DataLoader(MRIDataset(test_df, eval_tf), batch_size=BATCH_SIZE * 2, num_workers=workers)
print("batches/epoch:", len(train_dl))

## 7 · Model: EfficientNet-B0 pretrained on ImageNet

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=None if TEST_MODE else EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
steps = EPOCHS * len(train_dl)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=steps, pct_start=0.15)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

## 8 · Train

In [ ]:
import time, copy
from sklearn.metrics import f1_score

@torch.no_grad()
def predict(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            P.append(torch.softmax(model(x.to(DEVICE)), 1).float().cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

best, best_f1, history = None, -1, []
for ep in range(EPOCHS if not TEST_MODE else 1):
    model.train(); t0 = time.time(); loss_sum = n = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(y); n += len(y)
    vp, vy = predict(val_dl)
    vacc = (vp.argmax(1) == vy).mean(); vf1 = f1_score(vy, vp.argmax(1), average="macro")
    history.append((ep + 1, loss_sum / n, vacc, vf1))
    print(f"epoch {ep+1}: loss {loss_sum/n:.3f} | val acc {vacc:.3f} | val macro-F1 {vf1:.3f} | {time.time()-t0:.0f}s")
    if vf1 > best_f1: best_f1, best = vf1, copy.deepcopy(model.state_dict())
model.load_state_dict(best)

## 9 · Honest test results (patients the model has never seen)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt

tp, ty = predict(test_dl)
slice_acc = accuracy_score(ty, tp.argmax(1))
slice_f1 = f1_score(ty, tp.argmax(1), average="macro")

# Scan-level: average all slices of one MRI visit (how the app is used clinically)
t = test_df.reset_index(drop=True).copy()
t[[f"p{i}" for i in range(4)]] = tp
scan = t.groupby("scan").agg(label=("label", "first"), **{f"p{i}": (f"p{i}", "mean") for i in range(4)})
scan_pred = scan[[f"p{i}" for i in range(4)]].values.argmax(1)
scan_acc = accuracy_score(scan.label, scan_pred)

present = sorted(set(ty))
print(f"Slice-level accuracy: {slice_acc:.1%}   macro-F1: {slice_f1:.3f}")
print(f"Scan-level  accuracy: {scan_acc:.1%}   ({len(scan)} test scans, {t.patient.nunique()} patients)")
print(classification_report(ty, tp.argmax(1), labels=present, target_names=[CLASSES[i] for i in present], zero_division=0))

cm = confusion_matrix(ty, tp.argmax(1), labels=range(4))
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4), [c.replace(" ", "\n") for c in CLASSES], fontsize=8); ax.set_yticks(range(4), CLASSES, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Test confusion matrix (unseen patients)")
for i in range(4):
    for j in range(4): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150); plt.show()

## 10 · Export for the EDRAS backend

In [ ]:
import json, datetime, onnx, onnxruntime as ort

class WithSoftmax(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), dim=1)

export_model = WithSoftmax(model.float().cpu()).eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = f"{OUT_DIR}/dementia_cnn.onnx"
torch.onnx.export(export_model, dummy, onnx_path, input_names=["image"], output_names=["probs"],
                  dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(onnx_path))

# Verify ONNX output matches PyTorch (export can flip the model back to train mode, so reset it)
export_model.eval()
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
check = torch.randn(4, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    diff = np.abs(sess.run(None, {"image": check.numpy()})[0] - export_model(check).numpy()).max()
print("ONNX vs PyTorch max difference:", diff)
assert diff < 1e-3

per_class_recall = {CLASSES[i]: float(cm[i, i] / cm[i].sum()) for i in range(4) if cm[i].sum()}
meta = {
    "model": "EfficientNet-B0 (ImageNet pretrained, fine-tuned)",
    "classes": CLASSES, "severity": SEVERITY,
    "img_size": IMG_SIZE, "mean": MEAN, "std": STD, "grayscale_to_rgb": True,
    "metrics": {
        "test_slice_accuracy": round(float(slice_acc), 4),
        "test_slice_macro_f1": round(float(slice_f1), 4),
        "test_scan_accuracy": round(float(scan_acc), 4),
        "test_patients": int(t.patient.nunique()), "test_scans": int(len(scan)), "test_images": int(len(t)),
        "per_class_recall": per_class_recall,
        "split": "patient-wise (no patient in both train and test)",
    },
    "datasets": [OASIS_DATASET] + ([EXTRA_DATASET] if USE_EXTRA else []),
    "slice_range": list(SLICE_RANGE),
    "trained_at": datetime.datetime.utcnow().isoformat() + "Z",
    "history": [{"epoch": e, "loss": round(l, 4), "val_acc": round(float(a), 4), "val_f1": round(float(f), 4)} for e, l, a, f in history],
}
json.dump(meta, open(f"{OUT_DIR}/dementia_cnn_meta.json", "w"), indent=2)
print(json.dumps(meta["metrics"], indent=2))
print("Model size: %.1f MB" % (os.path.getsize(onnx_path) / 1e6))

## 11 · Save some unseen test scans (for trying the website)
These are slices from patients the model never trained on, a fair way to demo EDRAS.

In [ ]:
import shutil
demo_dir = Path(OUT_DIR) / "demo_test_scans"
demo_dir.mkdir(exist_ok=True)
for lab in range(4):
    sub = test_df[test_df.label == lab]
    if len(sub) == 0: continue
    for _, r in sub.sample(min(3, len(sub)), random_state=SEED).iterrows():
        shutil.copy(r.path, demo_dir / f"{CLASSES[lab].replace(' ', '_')}__{Path(r.path).name}")
print(sorted(p.name for p in demo_dir.iterdir()))

## 12 · Download

In [ ]:
archive = shutil.make_archive("/content/edras_trained_model" if not TEST_MODE else f"{OUT_DIR}/../edras_trained_model", "zip", OUT_DIR)
print("Created", archive)
if not TEST_MODE:
    from google.colab import files
    files.download(archive)

### Next
Send the downloaded `edras_trained_model.zip` back in the chat (or put `dementia_cnn.onnx` and
`dementia_cnn_meta.json` into `edras-system/ai_engine/models/`). The backend automatically uses the CNN when those
files exist, and the dashboard shows the real test accuracy from `dementia_cnn_meta.json`.